# Pipeline MCU — DuckDB → AuraDB (Colab & GitHub Actions)

Ce notebook exécute la pipeline complète de données Marvel :
1. Scraping Wikipedia pour les films MCU sortis
2. Téléchargement des données IMDb (TSV)
3. Construction d'une base DuckDB in-memory
4. Génération des CSV Marvel
5. Chargement dans Neo4j AuraDB

> **Credentials** : voir [docs/colab-credentials-guide.md](https://github.com/tarikbc/MoviesDB/blob/main/docs/colab-credentials-guide.md)

## Schéma de la pipeline

```mermaid
flowchart TD
    WEB["🌐 Wikipedia\nListe films MCU"]
    IMDB["📦 IMDb\ndatasets.imdbws.com\n(TSV .gz)"]
    WIKI["read_marvel_wiki.py"]
    FILMS["marvel_films\n(film, release_year)"]
    TSV["Fichiers TSV\nname.basics\ntitle.basics\ntitle.principals\ntitle.ratings\ntitle.episode"]
    DUCK[("🦆 DuckDB\nin-memory")]
    CSV["CSV Marvel\nactors / movies\ncharacters / relations"]
    AURA[("🔵 Neo4j AuraDB")]
    PIPE["load_into_auradb.py"]
    APP["🎬 Streamlit App\nmcugraph.streamlit.app"]

    WEB --> WIKI
    WIKI --> FILMS
    IMDB --> TSV
    TSV --> DUCK
    FILMS --> DUCK
    DUCK --> CSV
    CSV --> PIPE
    PIPE --> AURA
    AURA --> APP
```

In [ ]:
# ── 1. Installation des dépendances ──────────────────────────────────────────
import subprocess, sys

packages = [
    "duckdb",
    "neo4j",
    "pandas",
    "requests",
    "lxml",
    "python-dotenv",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("✅ Dépendances installées.")

In [ ]:
# ── 2. Téléchargement des scripts pipeline depuis le repo public ──────────────
import os
import urllib.request
from pathlib import Path

REPO_RAW = "https://raw.githubusercontent.com/tarikbc/MoviesDB/main/pipeline"
PIPELINE_DIR = Path("/content/pipeline")
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

scripts = [
    "read_marvel_wiki.py",
    "load_into_auradb.py",
    "pipeline_logger.py",
]

for script in scripts:
    dest = PIPELINE_DIR / script
    if not dest.exists():
        urllib.request.urlretrieve(f"{REPO_RAW}/{script}", dest)
        print(f"✅ {script} téléchargé.")
    else:
        print(f"⏭️  {script} déjà présent.")

sys.path.insert(0, str(PIPELINE_DIR))
print("✅ Scripts pipeline prêts.")

In [ ]:
# ── 3. Configuration des credentials AuraDB ───────────────────────────────────
# Détecte automatiquement l'environnement :
#   - Google Colab  : lit depuis les Colab Secrets (icône 🔑 barre latérale)
#   - GitHub Actions: lit depuis les variables d'environnement injectées par papermill

import os

def _get_secret(key: str, fallback: str = "") -> str:
    """Lit un secret depuis Colab Secrets ou les variables d'environnement."""
    try:
        from google.colab import userdata  # noqa: PLC0415
        return userdata.get(key) or os.getenv(key, fallback)
    except (ImportError, Exception):
        return os.getenv(key, fallback)

NEO4J_URI      = _get_secret("NEO4J_URI")
NEO4J_USERNAME = _get_secret("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = _get_secret("NEO4J_PASSWORD")
NEO4J_DATABASE = _get_secret("NEO4J_DATABASE", "neo4j")

if not NEO4J_URI or not NEO4J_PASSWORD:
    raise EnvironmentError(
        "❌ Credentials AuraDB manquants.\n"
        "   → Sur Colab : ajoutez NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD "
        "dans les Colab Secrets (🔑 barre latérale gauche).\n"
        "   → Sur GitHub Actions : configurez les secrets du dépôt.\n"
        "   → Voir docs/colab-credentials-guide.md pour le détail."
    )

# Injecter dans l'environnement pour les sous-processus (load_into_auradb.py)
os.environ["NEO4J_URI"]      = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["NEO4J_DATABASE"] = NEO4J_DATABASE

print(f"✅ Credentials chargés — URI : {NEO4J_URI[:30]}...")

In [ ]:
# ── 4. Téléchargement des données IMDb ────────────────────────────────────────
import gzip
import requests
from pathlib import Path

RAW_DIR = Path("/content/data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

IMDB_FILES = [
    "name.basics.tsv.gz",
    "title.basics.tsv.gz",
    "title.episode.tsv.gz",
    "title.principals.tsv.gz",
    "title.ratings.tsv.gz",
]
IMDB_BASE = "https://datasets.imdbws.com"

for fname in IMDB_FILES:
    gz_path  = RAW_DIR / fname
    tsv_path = RAW_DIR / fname.replace(".gz", "")

    if tsv_path.exists():
        print(f"⏭️  {tsv_path.name} déjà présent.")
        continue

    print(f"⬇️  Téléchargement de {fname}...", end=" ", flush=True)
    r = requests.get(f"{IMDB_BASE}/{fname}", stream=True, timeout=120)
    r.raise_for_status()
    with open(gz_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)

    print("extraction...", end=" ", flush=True)
    with gzip.open(gz_path, "rb") as gz, open(tsv_path, "wb") as out:
        out.write(gz.read())
    gz_path.unlink()
    print(f"✅ {tsv_path.name}")

print("✅ Données IMDb prêtes.")

In [ ]:
# ── 5. Création du schéma DuckDB et chargement des TSV ───────────────────────
import duckdb

con = duckdb.connect()  # base in-memory

RAW = str(RAW_DIR)

con.execute("""
    CREATE TABLE titles (
        title_id         VARCHAR NOT NULL,
        title_type       VARCHAR,
        primary_title    VARCHAR,
        original_title   VARCHAR,
        is_adult         BOOLEAN,
        start_year       INTEGER,
        end_year         INTEGER,
        runtime_minutes  INTEGER,
        genres           VARCHAR
    )
""")
con.execute("""
    CREATE TABLE people (
        person_id    VARCHAR NOT NULL,
        person_name  VARCHAR NOT NULL,
        born         SMALLINT,
        died         SMALLINT
    )
""")
con.execute("""
    CREATE TABLE crew (
        title_id         VARCHAR NOT NULL,
        person_id        VARCHAR NOT NULL,
        category         VARCHAR,
        job              VARCHAR,
        show_characters  VARCHAR
    )
""")
con.execute("""
    CREATE TABLE ratings (
        title_id  VARCHAR NOT NULL,
        rating    FLOAT,
        votes     INTEGER
    )
""")
con.execute("""
    CREATE TABLE episodes (
        episode_title_id  VARCHAR NOT NULL,
        show_title_id     VARCHAR NOT NULL,
        season_number     INTEGER,
        episode_number    INTEGER
    )
""")
con.execute("""
    CREATE TABLE marvel_films (
        film          VARCHAR NOT NULL,
        release_year  INTEGER NOT NULL
    )
""")
print("✅ Schéma DuckDB créé.")

print("⬇️  title.basics.tsv...", end=" ", flush=True)
con.execute(f"""
    INSERT INTO titles
    SELECT
        tconst,
        titleType,
        primaryTitle,
        originalTitle,
        TRY_CAST(isAdult AS BOOLEAN),
        TRY_CAST(startYear AS INTEGER),
        TRY_CAST(endYear AS INTEGER),
        TRY_CAST(runtimeMinutes AS INTEGER),
        genres
    FROM read_csv('{RAW}/title.basics.tsv',
        delim='\t', header=true, nullstr='\\N', quote='')
""")
print(f"✅ {con.execute('SELECT count(*) FROM titles').fetchone()[0]:,} lignes")

print("⬇️  name.basics.tsv...", end=" ", flush=True)
con.execute(f"""
    INSERT INTO people
    SELECT
        nconst,
        primaryName,
        TRY_CAST(birthYear AS SMALLINT),
        TRY_CAST(deathYear AS SMALLINT)
    FROM read_csv('{RAW}/name.basics.tsv',
        delim='\t', header=true, nullstr='\\N', quote='')
""")
print(f"✅ {con.execute('SELECT count(*) FROM people').fetchone()[0]:,} lignes")

print("⬇️  title.principals.tsv...", end=" ", flush=True)
con.execute(f"""
    INSERT INTO crew
    SELECT
        tconst,
        nconst,
        category,
        job,
        characters
    FROM read_csv('{RAW}/title.principals.tsv',
        delim='\t', header=true, nullstr='\\N', quote='')
""")
print(f"✅ {con.execute('SELECT count(*) FROM crew').fetchone()[0]:,} lignes")

print("⬇️  title.ratings.tsv...", end=" ", flush=True)
con.execute(f"""
    INSERT INTO ratings
    SELECT
        tconst,
        TRY_CAST(averageRating AS FLOAT),
        TRY_CAST(numVotes AS INTEGER)
    FROM read_csv('{RAW}/title.ratings.tsv',
        delim='\t', header=true, nullstr='\\N', quote='')
""")
print(f"✅ {con.execute('SELECT count(*) FROM ratings').fetchone()[0]:,} lignes")

print("⬇️  title.episode.tsv...", end=" ", flush=True)
con.execute(f"""
    INSERT INTO episodes
    SELECT
        tconst,
        parentTconst,
        TRY_CAST(seasonNumber AS INTEGER),
        TRY_CAST(episodeNumber AS INTEGER)
    FROM read_csv('{RAW}/title.episode.tsv',
        delim='\t', header=true, nullstr='\\N', quote='')
""")
print(f"✅ {con.execute('SELECT count(*) FROM episodes').fetchone()[0]:,} lignes")

print("\n✅ Toutes les données IMDb chargées dans DuckDB.")

In [ ]:
# ── 6. Scraping Wikipedia → marvel_films ──────────────────────────────────────
from read_marvel_wiki import main as scrape_marvel_films

marvel_df = scrape_marvel_films()
print(f"✅ {len(marvel_df)} films MCU récupérés depuis Wikipedia.")
print(marvel_df[["film", "release_year"]].head(5).to_string(index=False))

con.execute("DELETE FROM marvel_films")
con.register("marvel_films_df", marvel_df[["film", "release_year"]])
con.execute("INSERT INTO marvel_films SELECT film, release_year FROM marvel_films_df")
print(f"✅ {con.execute('SELECT count(*) FROM marvel_films').fetchone()[0]} films chargés dans DuckDB.")

In [ ]:
# ── 7. Correction manuelle Iron Man (équivalent imdb_add_character.sql) ────────
import json as _json

IRON_MAN_FIXES = [
    ("tt0371746", "nm0000375"),  # Iron Man (2008)
    ("tt1228705", "nm0000375"),  # Iron Man 2 (2010)
    ("tt1300854", "nm0000375"),  # Iron Man 3 (2013)
]

for title_id, person_id in IRON_MAN_FIXES:
    row = con.execute(
        "SELECT show_characters FROM crew WHERE title_id = ? AND person_id = ?",
        [title_id, person_id]
    ).fetchone()
    if row:
        chars = _json.loads(row[0]) if row[0] and row[0].startswith("[") else []
        if "Iron Man" not in chars:
            chars.append("Iron Man")
            con.execute(
                "UPDATE crew SET show_characters = ? WHERE title_id = ? AND person_id = ?",
                [_json.dumps(chars), title_id, person_id]
            )
            print(f"✅ Iron Man ajouté pour {title_id}")
        else:
            print(f"⏭️  Iron Man déjà présent pour {title_id}")

print("✅ Corrections Iron Man appliquées.")

In [ ]:
# ── 8. Génération de la table marvel_movies ────────────────────────────────────
con.execute("DROP TABLE IF EXISTS marvel_movies")
con.execute("""
    CREATE TABLE marvel_movies AS
    SELECT
        t.title_id,
        t.primary_title,
        t.genres,
        t.start_year,
        t.runtime_minutes
    FROM titles t
    INNER JOIN marvel_films m
        ON lower(t.primary_title) = lower(m.film)
        AND t.start_year = m.release_year
    WHERE
        t.title_type IN ('movie', 'tv_series')
        AND t.start_year IS NOT NULL
        AND t.genres IS NOT NULL
        AND t.runtime_minutes IS NOT NULL
    ORDER BY t.start_year
""")
n = con.execute("SELECT count(*) FROM marvel_movies").fetchone()[0]
print(f"✅ {n} films dans marvel_movies.")
con.execute("SELECT title_id, primary_title, start_year FROM marvel_movies ORDER BY start_year").df()

In [ ]:
# ── 9. Génération des CSV Marvel ──────────────────────────────────────────────
import pandas as pd

TESTS_DIR = Path("/content/data/tests")
TESTS_DIR.mkdir(parents=True, exist_ok=True)

# ── marvel_movies.csv ──────────────────────────────────────────────────────────
movies_df = con.execute("""
    SELECT title_id, primary_title, genres, start_year, runtime_minutes
    FROM marvel_movies
""").df()
movies_df.to_csv(TESTS_DIR / "marvel_movies.csv", index=False)
print(f"✅ marvel_movies.csv — {len(movies_df)} lignes")

# ── marvel_actors.csv ──────────────────────────────────────────────────────────
actors_df = con.execute("""
    SELECT DISTINCT p.person_id, p.person_name, p.born, p.died
    FROM people p
    INNER JOIN crew c ON p.person_id = c.person_id
    INNER JOIN marvel_movies m ON c.title_id = m.title_id
    WHERE lower(c.category) IN ('actor', 'actress')
    ORDER BY p.person_id
""").df()
actors_df.to_csv(TESTS_DIR / "marvel_actors.csv", index=False)
print(f"✅ marvel_actors.csv — {len(actors_df)} lignes")

# ── marvel_characters.csv ─────────────────────────────────────────────────────
crew_raw = con.execute("""
    SELECT c.title_id, c.show_characters
    FROM crew c
    INNER JOIN marvel_movies m ON c.title_id = m.title_id
    WHERE c.show_characters IS NOT NULL
""").df()

crew_raw["char_list"] = crew_raw["show_characters"].apply(
    lambda x: _json.loads(x) if x and x.startswith("[") else []
)
chars_df = (
    crew_raw.explode("char_list")
    .dropna(subset=["char_list"])
    .assign(character_name=lambda df: df["char_list"].str.strip().str.title())
    [["character_name"]]
    .drop_duplicates()
    .sort_values("character_name")
)
chars_df.to_csv(TESTS_DIR / "marvel_characters.csv", index=False)
print(f"✅ marvel_characters.csv — {len(chars_df)} lignes")

# ── marvel_person_plays_character.csv ─────────────────────────────────────────
crew_plays = con.execute("""
    SELECT DISTINCT p.person_id, c.show_characters
    FROM people p
    INNER JOIN crew c ON p.person_id = c.person_id
    INNER JOIN marvel_movies m ON c.title_id = m.title_id
    WHERE
        c.show_characters IS NOT NULL
        AND c.show_characters NOT LIKE '%Self%'
        AND lower(c.category) != 'self'
        AND c.category NOT LIKE 'archive_%'
""").df()

crew_plays["char_list"] = crew_plays["show_characters"].apply(
    lambda x: _json.loads(x) if x and x.startswith("[") else []
)
plays_df = (
    crew_plays.explode("char_list")
    .dropna(subset=["char_list"])
    .assign(character_name=lambda df: df["char_list"].str.strip())
    .query("character_name != '' and character_name.str.contains('Self') == False")
    [["person_id", "character_name"]]
    .drop_duplicates()
    .sort_values(["person_id", "character_name"])
)
plays_df.to_csv(TESTS_DIR / "marvel_person_plays_character.csv", index=False)
print(f"✅ marvel_person_plays_character.csv — {len(plays_df)} lignes")

# ── marvel_character_appears_in_movie.csv ─────────────────────────────────────
crew_appears = con.execute("""
    SELECT DISTINCT c.title_id, c.show_characters
    FROM crew c
    INNER JOIN marvel_movies m ON c.title_id = m.title_id
    WHERE c.show_characters IS NOT NULL
""").df()

crew_appears["char_list"] = crew_appears["show_characters"].apply(
    lambda x: _json.loads(x) if x and x.startswith("[") else []
)
appears_df = (
    crew_appears.explode("char_list")
    .dropna(subset=["char_list"])
    .assign(character_name=lambda df: df["char_list"].str.strip())
    .query("character_name != ''")
    [["title_id", "character_name"]]
    .drop_duplicates()
    .sort_values(["title_id", "character_name"])
)
appears_df.to_csv(TESTS_DIR / "marvel_character_appears_in_movie.csv", index=False)
print(f"✅ marvel_character_appears_in_movie.csv — {len(appears_df)} lignes")

# ── marvel_person_directs_movie.csv ───────────────────────────────────────────
directs_df = con.execute("""
    SELECT m.title_id, p.person_id
    FROM crew c
    INNER JOIN marvel_movies m ON c.title_id = m.title_id
    INNER JOIN people p ON c.person_id = p.person_id
    WHERE lower(c.category) = 'director'
      AND c.show_characters IS NULL
    ORDER BY m.title_id
""").df()
directs_df.to_csv(TESTS_DIR / "marvel_person_directs_movie.csv", index=False)
print(f"✅ marvel_person_directs_movie.csv — {len(directs_df)} lignes")

# ── marvel_person_produces_movie.csv ──────────────────────────────────────────
produces_df = con.execute("""
    SELECT m.title_id, p.person_id
    FROM crew c
    INNER JOIN marvel_movies m ON c.title_id = m.title_id
    INNER JOIN people p ON c.person_id = p.person_id
    WHERE lower(c.category) = 'producer'
      AND c.show_characters IS NULL
    ORDER BY m.title_id
""").df()
produces_df.to_csv(TESTS_DIR / "marvel_person_produces_movie.csv", index=False)
print(f"✅ marvel_person_produces_movie.csv — {len(produces_df)} lignes")

print(f"\n✅ Tous les CSV Marvel générés dans {TESTS_DIR}")

In [ ]:
# ── 10. Chargement dans Neo4j AuraDB ──────────────────────────────────────────
# Les credentials sont déjà dans os.environ (cellule 3).
# load_into_auradb.py utilise Path(__file__).parents[1]/data/tests
# → /content/data/tests — c'est là où les CSV ont été sauvegardés.

import subprocess

print("🚀 Lancement de load_into_auradb.py...")
result = subprocess.run(
    [sys.executable, str(PIPELINE_DIR / "load_into_auradb.py")],
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(f"❌ load_into_auradb.py a échoué (code {result.returncode})")

print("✅ Chargement AuraDB terminé.")

In [ ]:
# ── 11. Vérification ──────────────────────────────────────────────────────────
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with driver.session(database=NEO4J_DATABASE) as session:
    counts = session.run("""
        MATCH (n)
        RETURN labels(n)[0] AS label, count(n) AS total
        ORDER BY total DESC
    """).data()

    rels = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS rel_type, count(r) AS total
        ORDER BY total DESC
    """).data()

driver.close()

print("── Nœuds ─────────────────────────────")
for row in counts:
    print(f"  {row['label']:20s} {row['total']:>6,}")

print("\n── Relations ────────────────────────")
for row in rels:
    print(f"  {row['rel_type']:20s} {row['total']:>6,}")

print("\n✅ AuraDB prête — pipeline complète !")